# Verifying adjoint superposition against the Theis solution

Drawdown at a point is the sum of the drawdowns caused by each pumping well
acting on its own. That is the principle of **superposition**, and it is what
lets an analytical solution such as the Theis equation add up the effect of a
well field.

This notebook takes each well's response from an **adjoint sensitivity**
computed with [mf6adj](https://github.com/INTERA-Inc/mf6adj) instead of from a
formula, and checks the result against a problem whose answer is already known.
The **Theis** solution gives the drawdown around a well in a confined aquifer,
and it is itself built on superposition: several wells are added together in
space, and a well that starts later is added in time.

So there are three ways to get the drawdown at a point in such an aquifer, and
all three should agree:

1. run MODFLOW 6 with the wells on and difference it against a run with them off,
2. superpose the adjoint sensitivities, or
3. evaluate the Theis equation.

By the end of this notebook you will be able to:

- compute the sensitivity of head at an observation point to each pumping well's
  rate,
- rebuild the drawdown history at several observation wells by superposing those
  sensitivities, and
- read the difference between the three answers, and say which part of it is the
  method and which part is the model's discretization.

Once the method is verified here, [`mf6-adj-drawdown3d`](mf6-adj-drawdown3d.ipynb)
applies it to a layered three-dimensional model of a real valley, where no
analytical solution is available.


## What the adjoint gives you

The obvious way to find out how much a model result depends on a parameter is to
change the parameter and run the model again. One run per parameter, and there
are as many parameters as there are cells, so that gets expensive fast.

The adjoint approach turns it around. You first name the one model output you
care about — the **performance measure**, which here is the head at a particular
cell at a particular time — and then run the flow equations *backward* through
the saved time steps once. That single backward solve returns how much that one
output would change for a change in **any** parameter in **any** cell, including
the pumping rate of every well.

A fuller description is in [`mf6-adj-capture`](mf6-adj-capture.ipynb), and the
method itself is described in Hayek and others (2025), *MF6-ADJ: A Non-Intrusive
Adjoint Sensitivity Capability for MODFLOW 6*, Groundwater 63(6), 874-888.


## The idea

That "how much would the output change" is a **derivative**, and a well's
pumping rate is one of the parameters it can be taken with respect to. Written
for the head $h$ at an observation well and the rate $Q_i$ of well $i$,

$$\frac{\partial h}{\partial Q_i},$$

it is the **unit response**: the head change that well causes per unit of
pumping. Multiply it by the rate the well actually pumps and add up the wells,
and you have the drawdown:

$$s(t) = -\sum_i \sum_{\tau} \frac{\partial h(t)}{\partial Q_i(\tau)}\, Q_i(\tau).$$

The inner sum over $\tau$ runs over the stress periods, because a rate applied
in an earlier period still affects the head later.

Adding the wells up like this only works if the aquifer responds in proportion:
pump twice as hard and the drawdown doubles, and two wells together draw the
water level down by the sum of what each would do alone. Groundwater flow behaves
that way when the aquifer stays fully saturated and the boundaries do not move,
which is what **linear** means here. Where the sum fails to reproduce the
simulated drawdown, the difference measures how far the model departs from it.


Import the packages this notebook uses, and locate the MODFLOW 6 executable and
shared library.


In [ ]:
import pathlib as pl

import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries

lib_name, mf6_exe = find_mf6_libraries()

## An aquifer that meets Theis's assumptions

Build an aquifer the Theis solution actually describes: one layer, confined
(so it stays fully saturated and no water table rises or falls), the same
everywhere, of even thickness, with wells screened over the whole thickness so
water reaches them from top to bottom, and a domain wide enough that its edges
are not felt at the observation wells.

Two numbers describe such an aquifer. **Transmissivity** (`T`) is how readily it
passes water sideways — conductivity times thickness. **Storativity** (`S`) is
how much water it gives up per unit of head it loses. Together they set how far
and how fast a cone of depression spreads, and they are the only aquifer
properties the Theis equation needs. The parameters
are in `mf6_adj_helpers.py`; the domain is 30 km across, twice the radius of
influence at the end of the simulation, and the perimeter is held at the
starting head, which is Theis's condition that drawdown vanishes far away.


In [ ]:
print(f"transmissivity T = {adjh.THEIS_T:,.0f} m2/d")
print(f"storativity    S = {adjh.THEIS_S:.4f}")
radius = np.sqrt(
    2.25 * adjh.THEIS_T * adjh.THEIS_NPER * adjh.THEIS_PERLEN / adjh.THEIS_S
)
print(
    f"radius of influence at {adjh.THEIS_NPER * adjh.THEIS_PERLEN:.0f} days: {radius:,.0f} m"
)
print(f"half-width of the domain:      {adjh.THEIS_HALF:,.0f} m")

for name, (x, y, q, start) in adjh.THEIS_WELLS.items():
    print(
        f"  well {name}: {q:8,.0f} m3/d from period {start + 1:2d}"
        f"   at ({x:7,.0f}, {y:7,.0f}) m"
    )

Run the model once with the wells pumping. Three wells start in different
stress periods, so the superposition has to get both the spatial and the
temporal part right.


In [ ]:
theis_ws = pl.Path("models/adj-theis")
theis_sim = adjh.theis_simulation(theis_ws, mf6_exe)
theis_sim.write_simulation(silent=True)
success, buff = theis_sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

theis_head = adjh.theis_period_heads(theis_ws)

chd, wel, share = adjh.theis_boundary_share(theis_ws)
print(f"pumped from the wells:      {wel:9,.0f} m3/d")
print(f"drawn from the perimeter:   {chd:9,.0f} m3/d  ({100 * share:.2f}%)")

**What to look for.** By the last period the perimeter is supplying a few per
cent of the pumped water, so the cone has reached it. That water comes from
beyond the observation wells, though, which all sit within 4 km of the pumping:
widening the domain until the perimeter gives up nothing at all changes the
drawdown at those wells by less than a tenth of a millimetre. The drawdown *at*
the perimeter is zero whatever the domain size, because that is the condition
imposed there, so it is the flow across it that is worth looking at.


### Swapping the well and the observation point

A pumping well and an observation point can be exchanged. The drawdown you would
measure at B while pumping at A is the same as the drawdown you would measure at
A if you pumped the same way at B instead. Groundwater flow is symmetric in that
sense, and the symmetry has a name: **reciprocity**.

That is worth exploiting here. Put the performance measure at each *well* rather
than at each observation point, and one backward solve returns that well's
drawdown response at every cell in the model at once — so adding observation
points costs nothing. Three wells over ten stress periods is thirty measures, and
the drawdown can be rebuilt from them anywhere: at the four observation wells
below, or across the whole grid.


In [ ]:
theis_measures = {}
for name, (x, y, _, _) in adjh.THEIS_WELLS.items():
    for kper in range(adjh.THEIS_NPER):
        cellid = adjh.theis_cell(x, y)
        theis_measures[f"{name}{kper:02d}"] = [
            (kper, adjh.THEIS_NSTP - 1, 0, *cellid, "head")
        ]

theis_file = adjh.write_adj_file(theis_ws, "theis.adj", theis_measures)
adj = mf6adj.Mf6Adj(
    theis_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(theis_ws),
)
adj.solve_forward_model()
adj.solve_adjoint()
adj.finalize()
print(f"solved {len(theis_measures)} performance measures")

Now superpose: multiply each well's response by the rate it pumped in each
period, and add up the wells and the periods. That is the sum written out in
*The idea* above, evaluated at each of the four observation points.


In [ ]:
theis_rates = adjh.theis_rates()
superposed = {name: np.zeros(adjh.THEIS_NPER) for name in adjh.THEIS_OBS}

for well in adjh.THEIS_WELLS:
    for kper_pm in range(adjh.THEIS_NPER):
        kernels = adjh.period_sensitivity(theis_ws, f"{well}{kper_pm:02d}", "wel6_q")
        for kper, sens in kernels.items():
            for obs, (x, y) in adjh.THEIS_OBS.items():
                cellid = (0, *adjh.theis_cell(x, y))
                superposed[obs][kper_pm] += -sens[cellid] * theis_rates[well][kper]

### Compare all three

Read the simulated drawdown at each observation well, evaluate the Theis
equation there, and put the three side by side.


In [ ]:
days = np.arange(1, adjh.THEIS_NPER + 1) * adjh.THEIS_PERLEN

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, constrained_layout=True)
for ax, (obs, (x, y)) in zip(axes.flat, adjh.THEIS_OBS.items(), strict=False):
    row, col = adjh.theis_cell(x, y)
    simulated_obs = -theis_head[:, row, col]
    analytical = adjh.theis_analytical(x, y)

    ax.plot(days, simulated_obs, "o-", color="k", label="MODFLOW 6")
    ax.plot(
        days, superposed[obs], "s--", color="tab:red", ms=5, label="superposed adjoint"
    )
    ax.plot(days, analytical, "-", color="tab:blue", lw=1.2, label="Theis")
    ax.invert_yaxis()
    ax.set_title(f"{obs} — {np.hypot(x, y):,.0f} m from well A", fontsize=10)
    ax.set_ylabel("drawdown (m)")
axes[1, 0].set_xlabel("time (days)")
axes[1, 1].set_xlabel("time (days)")
axes[0, 0].legend(fontsize=8)

worst_adjoint = worst_theis = 0.0
for obs, (x, y) in adjh.THEIS_OBS.items():
    row, col = adjh.theis_cell(x, y)
    simulated_obs = -theis_head[:, row, col]
    worst_adjoint = max(worst_adjoint, np.abs(superposed[obs] - simulated_obs).max())
    worst_theis = max(
        worst_theis, np.abs(adjh.theis_analytical(x, y) - simulated_obs).max()
    )
print(f"largest difference, superposed vs MODFLOW 6: {worst_adjoint:.2e} m")
print(f"largest difference, Theis vs MODFLOW 6:      {worst_theis:.4f} m")

**What to look for.** The three curves lie on top of one another at all four
observation wells, and the steps where wells B and C start are reproduced by
each of them.

The two differences are not the same kind of thing. The superposed adjoint
matches the model to about 1e-11 m — machine precision — because it *is* the
model's own response, rebuilt from the sensitivities rather than recomputed.
Nothing is approximated in that step, so nothing is lost.

Theis differs by up to about two centimetres, and that is the model's
discretization rather than anything wrong with either. MODFLOW 6 reports a head
averaged over a 500 m cell and advances in finite time steps, while Theis is a
point value in continuous time. The time step is what dominates here: raising
`THEIS_NSTP` from 5 to 20 halves the difference, while refining the grid or
widening the domain barely moves it. The difference is largest in the first
period, when the cone of depression is younger than the time step can resolve,
and it shrinks as the cone grows.

That is the check worth having. The adjoint superposition is exact against the
model, and the model converges on the analytical solution as it is refined.


## Recap

- The adjoint sensitivity of a head to a well's rate is that well's **unit
  response** — the drawdown it causes per unit of pumping.
- Multiplying each well's response by its rate and summing over wells and stress
  periods rebuilds the drawdown history at any point.
- **Reciprocity** — putting the performance measure at the well rather than at
  the observation point — means one backward solve per well returns that well's
  response everywhere, so any number of observation points costs nothing extra.
- On an aquifer that meets the **Theis** assumptions, the superposed
  sensitivities, the simulated drawdown, and the analytical solution all agree:
  the first two to machine precision, because the superposition rebuilds the
  model's own response rather than approximating it, and the third to the
  model's discretization error.
- That is what makes the method trustworthy on a model where no analytical
  solution exists, which is where [`mf6-adj-drawdown3d`](mf6-adj-drawdown3d.ipynb)
  takes it next.
